# Gene Expression Explorer - GSE186755

**Interactive tool for searching gene expression across hPSC-to-EC differentiation.**

Search by official gene symbol or common alias (CD31, OCT4, VEGFR2, etc.).
Genes are automatically categorized into biological groups and plotted separately.

**Dataset:** Zhu Y, Liu J, Wang J, et al. *Protein & Cell* (2025).
[GEO: GSE186755](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE186755)

---

## How to use this notebook

1. **Run all cells in order** (Runtime > Run all)
2. **Upload your FPKM file** when prompted (GSE186755_hPSC-EC_fpkm.txt)
3. **Type gene names** in the search box (official symbols or aliases like CD31, OCT4)
4. **Click Add** or press Enter to add a gene
5. **Use preset buttons** to add whole biological groups at once
6. **Click Plot** to generate the figure

## What it produces

- **Panel A:** Heatmap showing log2 fold-change vs hESC for each selected gene
- **Panels B+:** Line plots grouped by biological category, showing log2(FPKM+1)
  across the five differentiation stages (hESC, VMC, EPC-1, EPC-2, EC)

## Sharing with your team

Upload this notebook to Google Drive and share the link. Anyone with access
can open it in Colab, upload the FPKM file, and search genes - no installation required.

---
## Cell 1: Import libraries

These are the Python packages this notebook uses:

| Library | What it does |
|---------|-------------|
| pandas | Reads and manipulates the FPKM data table |
| numpy | Math operations (log2 transformation) |
| matplotlib | Core plotting engine (draws figures) |
| seaborn | Makes heatmaps easier with better defaults |
| google.colab.files | Handles file upload in Colab environment |
| ipywidgets | Creates interactive buttons, text boxes, and UI elements |

In [ ]:
# Import all required libraries
# These are all pre-installed in Google Colab - no pip install needed

import pandas as pd                    # Data manipulation
import numpy as np                     # Math (log2 transform)
import matplotlib.pyplot as plt        # Plotting
import matplotlib.gridspec as gridspec  # Multi-panel figure layout
import matplotlib.lines as mlines      # Custom legend entries
import seaborn as sns                   # Heatmap
from google.colab import files         # File upload dialog
import ipywidgets as widgets           # Interactive UI elements
from IPython.display import display, clear_output  # Control notebook output

print('All libraries loaded successfully')

---
## Cell 2: Gene aliases and biological groups

This cell defines two key configuration dictionaries:

### GENE_ALIASES
Maps common gene names to their official symbols used in the FPKM file.
For example, when you search for "CD31", the tool looks up this dictionary
and finds "PECAM1", which is the name in the data file.

### KNOWN_GROUPS
Predefined sets of genes that belong to the same biological pathway.
When you click a preset button (e.g. "+ Stemness"), all genes in that
group are added at once. Each group has a display name, gene list, and color.

**To customize:** Add your own aliases or groups by editing these dictionaries.

In [ ]:
# =====================================================================
# GENE ALIASES
# =====================================================================
# Format: 'ALIAS': 'OFFICIAL_SYMBOL'
# All aliases are case-insensitive (CD31 = cd31 = Cd31)
#
# To add your own: just add a new line like:
#   'MY_NICKNAME': 'OFFICIAL_GENE_NAME',

GENE_ALIASES = {
    # Endothelial markers
    'CD31': 'PECAM1',          # platelet endothelial cell adhesion molecule
    'CD144': 'CDH5',           # VE-cadherin
    'VE-CADHERIN': 'CDH5',
    'VEGFR2': 'KDR',           # VEGF receptor 2
    'FLK1': 'KDR',             # mouse name for KDR
    'VEGFR1': 'FLT1',          # VEGF receptor 1
    'TIE2': 'TEK',             # angiopoietin receptor
    'ENDOGLIN': 'ENG',         # CD105
    'CD105': 'ENG',
    'ENOS': 'NOS3',            # endothelial nitric oxide synthase
    'CD146': 'MCAM',
    'CD309': 'KDR',            # another CD number for VEGFR2
    'CD202B': 'TEK',           # another CD number for TIE2
    'CD141': 'THBD',           # thrombomodulin

    # Pluripotency
    'OCT4': 'POU5F1',          # the most common alias in stem cell biology
    'OCT3': 'POU5F1',

    # Transcription factors
    'ER71': 'ETV2',            # early endothelial TF
    'PU.1': 'SPI1',            # myeloid master TF
    'SCL': 'TAL1',             # hematopoietic TF
    'COUP-TFII': 'NR2F2',      # venous identity TF

    # Immune / IFN-gamma pathway
    'P65': 'RELA',             # NF-kB p65 subunit
    'NFKB': 'RELA',
    'NF-KB': 'RELA',
    'MHC-II': 'CIITA',         # MHC class II transactivator
    'MHCII': 'CIITA',
    'MHC2TA': 'CIITA',
    'HLA-DR': 'HLA-DRA',       # MHC class II alpha chain
    'B2-MICROGLOBULIN': 'B2M',  # MHC class I component

    # Mesoderm / smooth muscle / hematopoietic
    'BRACHYURY': 'T',          # primitive streak marker
    'SMA': 'ACTA2',            # smooth muscle actin
    'ALPHA-SMA': 'ACTA2',
    'SM22': 'TAGLN',           # smooth muscle marker
    'DESMIN': 'DES',
    'CD45': 'PTPRC',           # pan-leukocyte marker
    'CD41': 'ITGA2B',          # megakaryocyte marker
    'VON WILLEBRAND': 'VWF',
    'VEGF': 'VEGFA',
}


# =====================================================================
# BIOLOGICAL GROUPS
# =====================================================================
# Each group has:
#   'genes': list of official gene symbols in the FPKM file
#   'color': hex color code (all genes in a group share a color,
#            distinguished by line style and marker shape)
#
# To add a new group, copy one entry and modify it:
#   'My Group': {'genes': ['GENE1','GENE2'], 'color': '#FF0000'},

KNOWN_GROUPS = {
    'Stemness': {
        'genes': ['POU5F1','SOX2','NANOG'],
        'color': '#7209B7',    # purple
    },
    'EC markers': {
        'genes': ['PECAM1','CDH5','KDR','VWF'],
        'color': '#185FA5',    # blue
    },
    'EC TFs': {
        'genes': ['ETV2','ERG','FLI1'],
        'color': '#007F5F',    # green
    },
    'IFN-g pathway': {
        'genes': ['RELA','IRF1','STAT1','CIITA'],
        'color': '#E63946',    # red
    },
    'Smooth muscle': {
        'genes': ['ACTA2','TAGLN','CNN1','MYH11'],
        'color': '#80B918',    # yellow-green
    },
    'Hematopoietic': {
        'genes': ['RUNX1','SPI1','CEBPB','GATA1'],
        'color': '#F4A261',    # orange
    },
    'Mesoderm': {
        'genes': ['T','MESP1','MIXL1','EOMES'],
        'color': '#D4A373',    # tan
    },
}


# =====================================================================
# STAGE MAPPING
# =====================================================================
# Maps each column name in the FPKM file to its differentiation stage.
# This groups replicates together (hESC.rep1 and hESC.rep2 are both 'hESC').
#
# IF YOU USE A DIFFERENT DATASET: change these to match your column names.
# For example:
#   'Day0_rep1': 'Day0', 'Day0_rep2': 'Day0',
#   'Day3_rep1': 'Day3', 'Day3_rep2': 'Day3',

STAGE_MAP = {
    'hESC.rep1':'hESC', 'hESC.rep2':'hESC',     # undifferentiated
    'VMC.rep1':'VMC',   'VMC.rep2':'VMC',        # vascular mesoderm
    'EPC-1.rep1':'EPC-1','EPC-1.rep2':'EPC-1',   # early endo progenitor
    'EPC-2.rep1':'EPC-2','EPC-2.rep2':'EPC-2',   # late endo progenitor
    'EC.rep1':'EC',     'EC.rep2':'EC',           # mature endothelial
}

# Order on x-axis (left = undifferentiated, right = mature)
STAGE_ORDER = ['hESC','VMC','EPC-1','EPC-2','EC']

# All column names (derived from STAGE_MAP keys)
STAGE_COLS = list(STAGE_MAP.keys())

# Visual styling for distinguishing genes within the same group
# LINE_STYLES: '-' = solid, '--' = dashed, (0,(3,1,1,1)) = dash-dot
LINE_STYLES = ['-','--',(0,(3,1,1,1)),(0,(5,2))]

# MARKER_SHAPES: 'o'=circle, 's'=square, '^'=triangle, 'D'=diamond
MARKER_SHAPES = ['o','s','^','D','v','P']

# Human-readable labels for legend (genes not listed use their symbol as-is)
GENE_LABELS = {
    'POU5F1':'POU5F1 (OCT4)',
    'PECAM1':'PECAM1 (CD31)',
    'CDH5':'CDH5 (CD144)',
    'KDR':'KDR (VEGFR2)',
    'RELA':'RELA (NF-kB)',
}

print('Configuration loaded')
print(f'  {len(GENE_ALIASES)} aliases defined')
print(f'  {len(KNOWN_GROUPS)} biological groups defined')
print(f'  {len(STAGE_ORDER)} differentiation stages')

---
## Cell 3: Upload FPKM file

This cell opens a file upload dialog. Select your FPKM file:
**GSE186755_hPSC-EC_fpkm.txt**

The file should be:
- Tab-separated (.txt or .tsv)
- First column = gene symbols
- Remaining columns = FPKM values per sample

Download from [GEO GSE186755](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE186755)
if you haven't already.

In [ ]:
# Upload the FPKM file
# A file picker dialog will appear - select your .txt file

print('Upload your FPKM file (GSE186755_hPSC-EC_fpkm.txt):')
print()

uploaded = files.upload()  # Opens the upload dialog

# Get the filename (whatever was uploaded)
fname = list(uploaded.keys())[0]

# Read the tab-separated file into a pandas DataFrame
# sep='\t'     = columns separated by tabs
# index_col=0  = first column (gene names) becomes the row index
df = pd.read_csv(fname, sep='\t', index_col=0)

# Extract sorted list of all gene names for the search function
available_genes = sorted(df.index.tolist())

print(f'\nLoaded {len(available_genes):,} genes from {fname}')
print(f'Columns: {list(df.columns)}')
print(f'\nFirst 5 genes: {available_genes[:5]}')

---
## Cell 4: Helper functions

Three core functions that power the tool:

1. **resolve_gene()** - Takes what you typed (e.g. 'CD31') and finds the
   matching gene in the dataset (e.g. 'PECAM1'). Checks exact match first,
   then case-insensitive, then alias lookup.

2. **classify_gene()** - Assigns a gene to a biological group (e.g. PECAM1
   goes to 'EC markers'). Genes not in any predefined group go to 'Other'.

3. **plot_genes()** - The main plotting function. Takes the list of selected
   genes, generates a heatmap (Panel A) and line plots grouped by category
   (Panels B, C, D...).

In [ ]:
# =====================================================================
# FUNCTION 1: resolve_gene()
# =====================================================================
# Takes a user-typed name and finds the official symbol in the dataset.
#
# Search order:
#   1. Exact match (case-sensitive):  'PECAM1' -> 'PECAM1'
#   2. Case-insensitive match:        'pecam1' -> 'PECAM1'
#   3. Alias lookup:                  'CD31'   -> 'PECAM1'
#
# Returns the official symbol if found, None if not found.

def resolve_gene(name):
    upper = name.upper()

    # 1. Exact match in the dataset
    if name in available_genes:
        return name

    # 2. Case-insensitive match
    for g in available_genes:
        if g.upper() == upper:
            return g

    # 3. Alias lookup
    if upper in GENE_ALIASES:
        official = GENE_ALIASES[upper]
        if official in available_genes:
            return official

    return None  # Not found anywhere


# =====================================================================
# FUNCTION 2: classify_gene()
# =====================================================================
# Checks if a gene belongs to any predefined biological group.
# Returns (group_name, color) if found, or ('Other', gray) if not.
#
# This is how the tool automatically sorts genes into separate
# line plot panels without you having to specify which panel.

def classify_gene(gene):
    for gname, ginfo in KNOWN_GROUPS.items():
        if gene in ginfo['genes']:
            return gname, ginfo['color']
    return 'Other', '#888780'


# =====================================================================
# FUNCTION 3: plot_genes()
# =====================================================================
# The main plotting function. Takes the list of selected genes and:
#   1. Groups them by biological category
#   2. Computes log2 fold-change vs hESC for the heatmap
#   3. Computes log2(FPKM+1) for the line plots
#   4. Draws Panel A (heatmap) + one line plot panel per group

def plot_genes(selected):
    if not selected:
        print('No genes selected.')
        return

    # --- Extract gene list and subset the data ---
    gene_list = [s[0] for s in selected]  # Just the gene symbols
    sub = df.loc[df.index.isin(gene_list)].reindex(gene_list)

    # --- Calculate hESC baseline for fold-change ---
    # Average the two hESC replicates as the reference denominator
    hesc_cols = [c for c in sub.columns if c.startswith('hESC')]
    hesc_mean = sub[hesc_cols].mean(axis=1)

    # --- Group genes by biological category ---
    # Creates: {'Stemness': {'genes':['POU5F1','SOX2'], 'color':'#7209B7'}, ...}
    groups = {}
    for gene, label, color in selected:
        if label not in groups:
            groups[label] = {'genes': [], 'color': color}
        groups[label]['genes'].append(gene)

    # --- Figure layout ---
    # 1 heatmap panel + 1 line plot per biological group
    n_groups = len(groups)
    fig = plt.figure(figsize=(7.5, 3.5 + n_groups * 2.5))
    gs = gridspec.GridSpec(
        1 + n_groups, 1,  # rows = heatmap + N line plots
        figure=fig, hspace=0.55,
        height_ratios=[1.3] + [1]*n_groups,
        top=0.95, bottom=0.05, left=0.11, right=0.74)

    # =============================================================
    # PANEL A: HEATMAP
    # =============================================================
    # Shows log2((FPKM+1) / (mean_hESC_FPKM+1)) for each gene.
    # hESC columns appear white (zero change), red = up, blue = down.

    ax1 = fig.add_subplot(gs[0])

    # Compute fold-change for every sample vs hESC mean
    fc_mat = pd.DataFrame(index=sub.index, columns=sub.columns)
    for col in sub.columns:
        fc_mat[col] = np.log2((sub[col]+1)/(hesc_mean+1))
    fc_mat = fc_mat.astype(float)

    # Order columns by differentiation stage
    col_order = [c for s in STAGE_ORDER for c in STAGE_MAP if STAGE_MAP[c]==s]
    fc_mat = fc_mat[col_order]

    # Two-line labels: 'hESC\nrep1'
    col_labels = [f"{STAGE_MAP[c]}\n{c.split('.')[1]}" for c in col_order]

    # Symmetric color scale capped at 10
    vmax = min(np.abs(fc_mat.values).max(), 10)

    # Gene labels with aliases where available
    hm_labels = [GENE_LABELS.get(g,g) for g in gene_list]

    # Draw heatmap
    sns.heatmap(fc_mat, ax=ax1, cmap='RdBu_r', center=0,
        vmin=-vmax, vmax=vmax,
        linewidths=0.4, linecolor='white',
        xticklabels=col_labels, yticklabels=hm_labels,
        cbar_kws={'shrink':0.5,'label':'log2 FC vs hESC'})
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0, ha='center', fontsize=7)
    ax1.set_yticklabels(ax1.get_yticklabels(), rotation=0, fontsize=7)
    ax1.tick_params(left=False, bottom=False)
    ax1.set_title('A', loc='left', fontweight='bold', fontsize=10, pad=6)

    # =============================================================
    # PREPARE LINE PLOT DATA
    # =============================================================
    # Convert wide format to long format:
    #   Wide:  gene | hESC.rep1 | hESC.rep2 | VMC.rep1 | ...
    #   Long:  gene | stage | log2fpkm
    # Long format has one row per gene per sample, easier for plotting.

    long = []
    for col in sub.columns:
        stage = STAGE_MAP.get(col)
        if not stage: continue
        for gene in gene_list:
            if gene in sub.index:
                long.append({'gene':gene,'stage':stage,
                    'log2fpkm':np.log2(sub.loc[gene,col]+1)})
    long = pd.DataFrame(long)
    long['stage'] = pd.Categorical(long['stage'],
        categories=STAGE_ORDER, ordered=True)

    # Average replicates per stage
    means = long.groupby(['gene','stage'],observed=True)['log2fpkm'].mean().reset_index()

    # Map stage names to x-positions: hESC=0, VMC=1, etc.
    x_pos = {s:i for i,s in enumerate(STAGE_ORDER)}

    # =============================================================
    # LINE PLOT PANELS (one per biological group)
    # =============================================================
    # Genes within each group share a color but have different
    # line styles (solid, dashed, dash-dot) and marker shapes
    # (circle, square, triangle) to stay distinguishable.

    letters = 'BCDEFGHIJ'
    for pi,(glabel,ginfo) in enumerate(groups.items()):
        ax = fig.add_subplot(gs[1+pi])
        genes_g = ginfo['genes']

        for gi,gene in enumerate(genes_g):
            # Get mean values across replicates
            gm = means[means.gene==gene].sort_values('stage')
            xs = [x_pos[s] for s in gm.stage]
            ys = gm.log2fpkm.values

            # Draw connecting line (mean values)
            ax.plot(xs,ys,color=ginfo['color'],linewidth=1.6,
                linestyle=LINE_STYLES[gi%len(LINE_STYLES)],zorder=2)

            # Overlay individual replicate data points
            gr = long[long.gene==gene]
            for _,row in gr.iterrows():
                ax.scatter(x_pos[row.stage],row.log2fpkm,
                    color=ginfo['color'],s=30,zorder=3,
                    marker=MARKER_SHAPES[gi%len(MARKER_SHAPES)],
                    edgecolors='white',linewidths=0.5)

        # Format axes
        ax.set_xticks(range(len(STAGE_ORDER)))
        ax.set_xticklabels(STAGE_ORDER, fontsize=8)
        ax.set_ylabel('log2(FPKM+1)', fontsize=8)
        ax.set_xlim(-0.4, len(STAGE_ORDER)-0.6)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_title(letters[pi], loc='left', fontweight='bold', fontsize=10, pad=6)

        # Group name above plot
        ax.text(0.5,1.02,glabel,transform=ax.transAxes,
            ha='center',va='bottom',fontsize=9,fontstyle='italic',color='#5F5E5A')

        # Legend outside plot area
        labels = [GENE_LABELS.get(g,g) for g in genes_g]
        handles = [mlines.Line2D([],[],color=ginfo['color'],linewidth=1.6,
            linestyle=LINE_STYLES[i%len(LINE_STYLES)],
            marker=MARKER_SHAPES[i%len(MARKER_SHAPES)],markersize=5,
            markeredgecolor='white',markeredgewidth=0.5,label=lab)
            for i,lab in enumerate(labels)]
        ax.legend(handles=handles,frameon=False,fontsize=7.5,
            bbox_to_anchor=(1.02,1),loc='upper left',handlelength=2.2)

    plt.show()

print('All functions ready')

---
## Cell 5: Interactive search interface

This cell creates the interactive UI with:
- **Search box:** Type a gene name and press Enter or click Add
- **Preset buttons:** Click to add all genes in a biological group
- **Gene pills:** Shows which genes are currently selected
- **Plot button:** Generates the figure with all selected genes
- **Clear all:** Removes all selected genes

### Tips:
- Search is case-insensitive (cd31 = CD31 = Cd31)
- Aliases are resolved automatically (CD31 finds PECAM1)
- You can mix preset groups and individual genes
- Click Plot after each change to update the figure

In [ ]:
# =====================================================================
# INTERACTIVE UI
# =====================================================================
# Uses ipywidgets to create buttons and text boxes in the notebook.
#
# How it works:
#   - selected_genes: a list that accumulates (gene, group, color) tuples
#   - Each button has a callback function that modifies selected_genes
#   - The Plot button calls plot_genes() with the current selection
#   - gene_pills: HTML widget showing colored tags for selected genes

# This list stores all currently selected genes as (gene, group, color) tuples
selected_genes = []

# Output widget: this is where the plot will appear
output = widgets.Output()

# --- Create UI elements ---

# Text input for gene search
search_box = widgets.Text(
    placeholder='Type gene name (e.g. CD31, OCT4, CIITA)',
    description='Search:',
    layout=widgets.Layout(width='400px'))

# Action buttons
add_btn = widgets.Button(description='Add', button_style='primary')
clear_btn = widgets.Button(description='Clear all')
plot_btn = widgets.Button(description='Plot', button_style='success')

# Status messages (shows 'Added: PECAM1' or 'Not found' feedback)
status = widgets.HTML(value='')

# Colored gene tags showing current selection
gene_pills = widgets.HTML(value='<i>No genes selected</i>')


# --- Create preset group buttons ---
# One button per biological group (Stemness, EC markers, etc.)
# Clicking adds all genes in that group at once

group_buttons = []
for gname, ginfo in KNOWN_GROUPS.items():
    b = widgets.Button(
        description=f'+ {gname}',
        layout=widgets.Layout(width='auto'),
        style={'button_color': '#f0f0f0'})

    # make_cb creates a unique callback for each button
    # (Python closure trick - without this, all buttons would
    #  reference the same 'gname' variable)
    def make_cb(name):
        def cb(btn):
            ginfo = KNOWN_GROUPS[name]
            existing = {s[0] for s in selected_genes}
            added = []
            for g in ginfo['genes']:
                if g in available_genes and g not in existing:
                    selected_genes.append((g, name, ginfo['color']))
                    added.append(g)
            if added:
                status.value = f\"Added: {', '.join(added)}\"
            update_pills()
        return cb
    b.on_click(make_cb(gname))
    group_buttons.append(b)


# --- Callback functions ---

def update_pills():
    """Refresh the colored gene tags display."""
    if not selected_genes:
        gene_pills.value = '<i>No genes selected</i>'
        return
    html = ''
    for g, label, color in selected_genes:
        html += f'<span style="display:inline-block;padding:2px 8px;margin:2px;'
        html += f'border-radius:10px;background:{color}22;color:{color};'
        html += f'font-size:12px;font-weight:500;border:1px solid {color}44;">'
        html += f'{g}</span> '
    gene_pills.value = html


def on_add(btn):
    """Called when Add button is clicked or Enter is pressed."""
    name = search_box.value.strip()
    if not name:
        return

    # Try to resolve the gene name (direct match or alias)
    resolved = resolve_gene(name)
    if resolved is None:
        status.value = f'<span style="color:red">{name} not found</span>'
        return

    # Check for duplicates
    if resolved in [s[0] for s in selected_genes]:
        status.value = f'{resolved} already added'
        return

    # Classify into biological group and add
    label, color = classify_gene(resolved)
    selected_genes.append((resolved, label, color))

    # Show feedback with alias info if applicable
    alias = f' (alias for {resolved})' if name.upper() != resolved.upper() else ''
    status.value = f'Added: {resolved}{alias} [{label}]'
    search_box.value = ''  # Clear the search box
    update_pills()


def on_clear(btn):
    """Called when Clear all button is clicked."""
    selected_genes.clear()
    status.value = 'Cleared'
    update_pills()


def on_plot(btn):
    """Called when Plot button is clicked. Generates the figure."""
    with output:
        clear_output()         # Remove previous plot
        plot_genes(selected_genes)  # Generate new plot


def on_enter(change):
    """Called when Enter is pressed in the search box."""
    on_add(None)


# --- Connect callbacks to UI elements ---
# .on_click() = run this function when the button is clicked
# .on_submit() = run this function when Enter is pressed in the text box

add_btn.on_click(on_add)
clear_btn.on_click(on_clear)
plot_btn.on_click(on_plot)
search_box.on_submit(on_enter)
update_pills()


# --- Display the UI ---
# HBox = horizontal box (elements side by side)
# display() shows the widget in the notebook

display(widgets.HBox([search_box, add_btn, clear_btn, plot_btn]))
display(widgets.HBox(group_buttons))
display(status)
display(gene_pills)
display(output)

---
## Quick reference

### Supported aliases

| You type | Finds | Category |
|----------|-------|----------|
| CD31 | PECAM1 | EC markers |
| CD144, VE-cadherin | CDH5 | EC markers |
| OCT4 | POU5F1 | Stemness |
| VEGFR2, CD309 | KDR | EC markers |
| TIE2, CD202B | TEK | EC markers |
| CD105, endoglin | ENG | EC markers |
| alpha-SMA, SMA | ACTA2 | Smooth muscle |
| PU.1 | SPI1 | Hematopoietic |
| ER71 | ETV2 | EC TFs |
| P65, NF-KB | RELA | IFN-g pathway |
| MHC-II, MHC2TA | CIITA | IFN-g pathway |
| HLA-DR | HLA-DRA | IFN-g pathway |
| brachyury | T | Mesoderm |

### Adding your own aliases

Edit the GENE_ALIASES dictionary in Cell 2:

    'MY_ALIAS': 'OFFICIAL_GENE_NAME',

### Adding a new biological group

Edit KNOWN_GROUPS in Cell 2:

    'My Group': {'genes': ['GENE1','GENE2'], 'color': '#FF0000'},

### Using with a different dataset

Update STAGE_MAP and STAGE_ORDER in Cell 2 to match your column names.